<div align="center">

<a href="https://colab.research.google.com/github/utkukose/ai-applications-in-engineering-NB-lecture/blob/main/weeks/week-11/NB11_cnn_vision.ipynb"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open in Colab"></a>

# Week 11: Seeing with Networks: Convolutional Neural Networks and Computer Vision

**Artificial Intelligence Applications in Engineering (MUH-920 Mühendislikte Yapay Zeka Uygulamaları)**<br>Prof. Dr. Utku Kose<br>Full Professor, Department of Computer Engineering, Süleyman Demirel University, Isparta, Türkiye<br>Founding Director, AI Application and Research Center (YAZEM), Süleyman Demirel University<br>Additional affiliations: University of North Dakota (USA), Universidad Panamericana (Mexico City, Mexico), Vel Tech University (Chennai, India)<br>[ORCID 0000-0002-9652-6415](https://orcid.org/0000-0002-9652-6415) | [utkukose.com](https://www.utkukose.com) | [github.com/utkukose](https://github.com/utkukose)<br>[utkukose@sdu.edu.tr](mailto:utkukose@sdu.edu.tr) | [utku.kose@und.edu](mailto:utku.kose@und.edu) | [ukose@up.edu.mx](mailto:ukose@up.edu.mx) | [utkukose@gmail.com](mailto:utkukose@gmail.com)

<sub>Content licensed under CC BY 4.0, code under MIT. This course is updated in line with current developments in the field. Last update: October 2026.</sub>

</div>

## About this notebook

The notebook computes convolutions and pooling by hand, generates a dataset of concrete surface images with and without cracks, and trains a small convolutional network on it in PyTorch. Grad-CAM shows where the network looks [1]. It then compares a convolutional network with a multilayer perceptron on Fashion-MNIST [2], or on generated shapes when the download is unavailable, and fine-tunes a ResNet-18 pretrained on ImageNet on the crack images [3, 4].

This notebook is the hands-on part of Week 11. It opens with Python step 11: Images, batches and network classes and continues with the sections listed in the table of contents. The concepts are explained on the [lecture page](https://utkukose.github.io/ai-applications-in-engineering-NB-lecture/weeks/week-11/lecture.html) and in the [PDF notes](https://github.com/utkukose/ai-applications-in-engineering-NB-lecture/blob/main/weeks/week-11/Week11_Lecture_Notes.pdf). The [interactive lab](https://utkukose.github.io/ai-applications-in-engineering-NB-lecture/weeks/week-11/lab.html) holds the simulation, the self-assessment and the reflection, and the [week overview](https://github.com/utkukose/ai-applications-in-engineering-NB-lecture/blob/main/weeks/week-11/README.md) gives the learning outcomes, the study path, the discipline challenges and the tasks.

## How to use this notebook

Run the cells in order with Shift+Enter. No earlier Python experience is assumed: Python step 11 teaches the Python of the week first, and every further piece of Python appears in a cell marked **Python move**, which explains the syntax right where it is first needed. Exercise cells start with `None` placeholders and print feedback through `check(...)`. Reference solutions sit in collapsed cells: Open them only after a genuine attempt. Cells that download public data need an internet connection. When a source is unreachable, the notebook falls back to a documented synthetic stand-in so that every later cell still runs.

## 0. Setup

A GPU speeds up this notebook. In Colab, select Runtime, Change runtime type, and a GPU accelerator; the code also runs on a CPU, more slowly.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.utils.data import TensorDataset, DataLoader
from scipy.signal import correlate2d
from scipy.ndimage import gaussian_filter

plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})
torch.manual_seed(0)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", DEVICE)

def check(name, value, expected, tol=1e-6):
    """Compare an exercise answer with the reference value and print feedback."""
    if value is None:
        print(f"[{name}] Not attempted yet. Replace None with your answer and run the cell again.")
        return
    try:
        ok = abs(float(value) - float(expected)) <= tol * max(1.0, abs(float(expected)))
    except (TypeError, ValueError):
        ok = value == expected
    print(f"[{name}] {'Correct.' if ok else 'Not yet. Check the logic and run again.'}")

## Python step 11: Images, batches and network classes

This step teaches the Python of the week with the week's own example. Read each explanation, run the cell below it and compare the output with the text; then change a value and predict the new result before running the cell again. A quick check and short exercises with immediate feedback close the step. The [Python path](https://github.com/utkukose/ai-applications-in-engineering-NB-lecture/blob/main/PYTHON_PATH.md) shows how the fourteen steps of the course build on each other.

### Images are arrays

A greyscale image is a two-dimensional array of intensities, usually scaled to the range 0 to 1, with the shape (height, width). Colour images add a third axis for the red, green and blue channels. Everything from Weeks 4 and 9 applies: Slicing crops an image, reversing an axis mirrors it, and masks find the pixels that meet a condition. The practice image below is light grey with a dark vertical crack in column 4.

In [ ]:
import numpy as np

img = np.full((8, 10), 0.7)          # light grey image, 8 rows and 10 columns
img[:, 4] = 0.1                      # a dark crack in column 4
print(img.shape, img.min(), img.max())
crop = img[2:6, 3:6]                 # rows 2 to 5, columns 3 to 5
print(crop)
print("dark pixels:", int((img < 0.3).sum()))
flipped = img[:, ::-1]               # mirror from left to right
print("crack column after flipping:", int(np.argmin(flipped.mean(axis=0))))

The slice `::-1` steps backwards through an axis. Flips and small rotations are the data augmentation of this week: They create new training images that are still valid examples of a crack.

### From images to batches

Networks process several images at once. `np.stack` joins images of equal shape along a new first axis, which gives the shape (number of images, height, width). PyTorch convolution layers expect a channel axis in second place, (batch, channels, height, width), so greyscale images receive an axis of length one with `[:, None]`. Libraries that read colour image files usually return (height, width, channels), and `permute` reorders the axes of a tensor.

In [ ]:
import torch

batch = np.stack([img, flipped, img * 0.9])          # three images
print(batch.shape)
xb = torch.tensor(batch[:, None], dtype=torch.float32)
print(xb.shape)
rgb = torch.zeros(64, 64, 3)                          # colour image as stored in files
print(rgb.permute(2, 0, 1).shape)                     # channels first, as PyTorch expects

### Iterating in batches

A `DataLoader` serves a dataset in batches of a chosen size, optionally in shuffled order, and a `for` loop receives one batch of images and labels per pass. `enumerate` numbers the batches. The last batch is smaller when the batch size does not divide the number of examples.

In [ ]:
from torch.utils.data import TensorDataset, DataLoader

images = torch.rand(10, 1, 8, 10)
labels = torch.tensor([0, 1] * 5)
loader = DataLoader(TensorDataset(images, labels), batch_size=4, shuffle=False)
for i, (xb_i, yb_i) in enumerate(loader):
    print(i, tuple(xb_i.shape), yb_i.tolist())

The expression `[0, 1] * 5` repeats a list five times, the list behaviour that NumPy arrays replace with element-wise arithmetic.

### A network is a class

Week 6 introduced classes. A PyTorch network is a class that inherits from `nn.Module`: It gains all the machinery of a network and adds its own layers in `__init__` and its computation in `forward`. The call `super().__init__()` first runs the initialisation of the parent class. Calling the object, as in `net(xb)`, runs `forward`.

In [ ]:
from torch import nn


class TinyCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv = nn.Conv2d(1, 4, kernel_size=3, padding=1)
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Linear(4, 2)

    def forward(self, x):
        x = torch.relu(self.conv(x))
        return self.fc(self.pool(x).flatten(1))


net = TinyCNN()
print(net(xb).shape)
print(sum(p.numel() for p in net.parameters()), "parameters")

The convolution has 4 filters of 3 by 3 weights plus 4 biases, 40 parameters, and the linear layer has 4 times 2 weights plus 2 biases, 10 parameters.

**Quick check.** Which shape does PyTorch expect for a batch of 16 colour images of 32 by 32 pixels?

A. (16, 32, 32, 3)  
B. (16, 3, 32, 32)  
C. (3, 16, 32, 32)  
D. (32, 32, 3, 16)

<details><summary>Show the answer</summary>

**B.** Convolution layers expect batch, channels, height and width, in this order.

</details>

### Exercises for Python step 11

**Exercise 11.1.** Compute the mean intensity of `crop` and store it in `py_crop_mean`.

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
def _py11_crop_mean_reference():
    return (0.7 + 0.1 + 0.7) / 3

In [ ]:
py_crop_mean = None
check("Exercise 11.1", py_crop_mean, _py11_crop_mean_reference())

**Exercise 11.2.** Stack five greyscale images of 64 by 64 pixels, for example `np.zeros((64, 64))`, into a batch, add the channel axis, and store the resulting shape as a tuple in `py_batch_shape`.

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
def _py11_batch_shape_reference():
    return (5, 1, 64, 64)

In [ ]:
py_batch_shape = None
check("Exercise 11.2", py_batch_shape, _py11_batch_shape_reference())

**Exercise 11.3.** Change `TinyCNN` so that the convolution has 8 filters and the linear layer maps 8 features to 2 classes. Count the parameters of the new network and store the number in `py_params_8`.

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
def _py11_params_8_reference():
    return (8 * 1 * 3 * 3 + 8) + (8 * 2 + 2)

In [ ]:
py_params_8 = None
check("Exercise 11.3", py_params_8, _py11_params_8_reference())

## 1. Convolution by hand

Deep learning libraries implement convolution as cross-correlation: the filter is not flipped. `scipy.signal.correlate2d` computes the same operation.

### Your turn, exercise 1

Apply the vertical Sobel filter to the 5 by 5 patch below without padding and store the value of the centre of the 3 by 3 output in `centre_value`. Compute it with an explicit double loop, then check it with `correlate2d(patch, sobel, mode="valid")`. The patch contains a dark vertical crack in its middle column. Compare the centre value with the values to its left and right: An edge filter responds to the two edges of a thin crack with opposite signs, and the responses largely cancel on the crack itself.

In [ ]:
patch = np.array([[9, 9, 2, 9, 9],
                  [9, 9, 1, 9, 9],
                  [9, 8, 1, 9, 9],
                  [9, 9, 2, 8, 9],
                  [9, 9, 1, 9, 9]], dtype=float)
sobel = np.array([[-1, 0, 1], [-2, 0, 2], [-1, 0, 1]], dtype=float)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
def _centre_value_reference():
    return float(correlate2d(patch, sobel, mode="valid")[1, 1])

In [ ]:
centre_value = None   # your computation
check("Exercise 1", centre_value, _centre_value_reference())

### Your turn, exercise 2

A 128 by 128 input passes a convolution with kernel 5, padding 2 and stride 2, and then 2 by 2 max pooling. Use the formula (W - K + 2P) / S + 1 and store the final width in `final_width`.

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
def _final_width_reference():
    return ((128 - 5 + 2 * 2) // 2 + 1) // 2

In [ ]:
final_width = None   # your computation
check("Exercise 2", final_width, _final_width_reference())

## 2. A generated dataset of concrete surfaces

Each image is a 64 by 64 concrete-like texture made of smoothed noise and pores. Half of the images contain a crack drawn as a random walk with varying width and contrast. Generated data have one advantage for teaching: The true label of every image is known. They also have a limit: A model trained on them is not ready for real photographs, which have lighting, stains, joints and shadows.

**Python move: functions that return arrays.** `concrete_image` returns one image; a list comprehension builds the dataset, and `np.stack` turns the list into one array.

In [ ]:
def concrete_image(crack, rng, size=64):
    img = gaussian_filter(rng.normal(0, 1, (size, size)), 1.2) * 0.08 + 0.62 + rng.normal(0, 0.02)
    pores = rng.random((size, size)) < 0.004
    img[pores] -= rng.uniform(0.15, 0.35)
    if crack:
        vertical = rng.random() < 0.5
        pos, width = rng.uniform(10, size - 10), rng.choice([1, 1, 2])
        depth = rng.uniform(0.2, 0.45)
        for t in range(size):
            pos = np.clip(pos + rng.normal(0, 0.9), 3, size - 4)
            p = int(pos)
            if vertical:
                img[t, p:p + width] -= depth
            else:
                img[p:p + width, t] -= depth
    return np.clip(gaussian_filter(img, 0.5), 0, 1)


rng = np.random.default_rng(11)
labels = rng.integers(0, 2, 2400)
images = np.stack([concrete_image(bool(c), rng) for c in labels]).astype(np.float32)
fig, ax = plt.subplots(2, 6, figsize=(10, 3.6))
for a, img, lab in zip(ax.ravel(), images[:12], labels[:12]):
    a.imshow(img, cmap="gray", vmin=0, vmax=1); a.set_title("crack" if lab else "no crack", fontsize=8); a.axis("off")
plt.tight_layout(); plt.show()

## 3. A small convolutional network

Two convolutional blocks extract features, adaptive average pooling summarises each feature map, and a linear layer outputs two class scores.

In [ ]:
class CrackNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(nn.Conv2d(1, 8, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
                                      nn.Conv2d(8, 16, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
                                      nn.Conv2d(16, 32, 3, padding=1), nn.ReLU())
        self.head = nn.Sequential(nn.AdaptiveAvgPool2d(1), nn.Flatten(), nn.Linear(32, 2))

    def forward(self, x):
        return self.head(self.features(x))


X = torch.tensor(images).unsqueeze(1)                   # shape (N, 1, 64, 64)
y = torch.tensor(labels)
Xtr, ytr, Xte, yte = X[:2000], y[:2000], X[2000:], y[2000:]


def fit(model, Xtr, ytr, epochs=6, lr=2e-3, batch=64, augment=False):
    model.to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    loader = DataLoader(TensorDataset(Xtr, ytr), batch_size=batch, shuffle=True)
    for epoch in range(epochs):
        model.train(); total = 0.0
        for xb, yb in loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            if augment:                                  # random flips and quarter turns
                if torch.rand(1) < 0.5:
                    xb = torch.flip(xb, dims=[3])
                xb = torch.rot90(xb, k=int(torch.randint(0, 4, (1,))), dims=[2, 3])
            opt.zero_grad()
            loss = nn.functional.cross_entropy(model(xb), yb)
            loss.backward(); opt.step(); total += loss.item() * len(xb)
        print(f"epoch {epoch + 1}: training loss {total / len(Xtr):.4f}")
    return model


def accuracy(model, X, y, batch=256):
    model.eval(); correct = 0
    with torch.no_grad():
        for i in range(0, len(X), batch):
            correct += (model(X[i:i + batch].to(DEVICE)).argmax(1).cpu() == y[i:i + batch]).sum().item()
    return correct / len(X)


crack_net = fit(CrackNet(), Xtr, ytr, augment=True)
print("test accuracy:", accuracy(crack_net, Xte, yte))
print("parameters:", sum(p.numel() for p in crack_net.parameters()))

## 4. Where does the network look? Grad-CAM

**Python move: hooks.** A forward hook stores the output of the last convolutional layer, and its gradient with respect to the crack score is captured through `retain_grad`. Averaging the gradients over each feature map gives a weight per map; the weighted sum of the maps, passed through ReLU, is the heatmap [1].

In [ ]:
def grad_cam(model, img, target=1):
    model.eval()
    store = {}
    handle = model.features[-1].register_forward_hook(lambda m, i, o: store.update(maps=o))
    x = img.unsqueeze(0).to(DEVICE).requires_grad_(True)
    score = model(x)[0, target]
    store["maps"].retain_grad()
    score.backward()
    handle.remove()
    maps, grads = store["maps"][0], store["maps"].grad[0]
    cam = torch.relu((grads.mean(dim=(1, 2))[:, None, None] * maps).sum(0))
    cam = nn.functional.interpolate(cam[None, None], size=img.shape[-2:], mode="bilinear", align_corners=False)[0, 0]
    return (cam / (cam.max() + 1e-8)).detach().cpu().numpy()


idx = [i for i in range(len(Xte)) if yte[i] == 1][:4]
fig, ax = plt.subplots(2, 4, figsize=(9, 4.4))
for k, i in enumerate(idx):
    ax[0, k].imshow(Xte[i, 0], cmap="gray", vmin=0, vmax=1); ax[0, k].axis("off")
    ax[1, k].imshow(Xte[i, 0], cmap="gray", vmin=0, vmax=1); ax[1, k].imshow(grad_cam(crack_net, Xte[i]), cmap="jet", alpha=0.45); ax[1, k].axis("off")
ax[0, 0].set_title("test images", fontsize=9, loc="left"); ax[1, 0].set_title("Grad-CAM for 'crack'", fontsize=9, loc="left")
plt.tight_layout(); plt.show()

### Your turn, exercise 3

Count the parameters of `nn.Conv2d(3, 16, kernel_size=3)` including biases and store the number in `conv_params`. Then explain why the number does not depend on the image size.

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
def _conv_params_reference():
    return sum(p.numel() for p in nn.Conv2d(3, 16, kernel_size=3).parameters())

In [ ]:
conv_params = None   # your computation
check("Exercise 3", conv_params, _conv_params_reference())

## 5. Convolutional network against a multilayer perceptron

Fashion-MNIST contains 70,000 greyscale images of 28 by 28 pixels from ten clothing classes and was designed as a harder drop-in replacement for handwritten digits [2]. If it cannot be downloaded, a generated set of four shapes is used. Both networks train for two epochs on 12,000 images.

In [ ]:
try:
    from torchvision import datasets
    train = datasets.FashionMNIST("data", train=True, download=True)
    test = datasets.FashionMNIST("data", train=False, download=True)
    Xa = train.data[:12000].float().unsqueeze(1) / 255; ya = train.targets[:12000]
    Xb = test.data[:3000].float().unsqueeze(1) / 255; yb = test.targets[:3000]
    BENCH = "Fashion-MNIST"
except Exception as error:
    print("Download failed, generating shapes instead:", type(error).__name__)
    r = np.random.default_rng(3)
    def shape(k):
        im = np.zeros((28, 28), np.float32); cy, cx, s = r.integers(9, 19, 2).tolist() + [int(r.integers(4, 8))]
        yy, xx = np.mgrid[:28, :28]
        if k == 0: im[(yy - cy) ** 2 + (xx - cx) ** 2 <= s * s] = 1
        elif k == 1: im[max(cy - s, 0):cy + s, max(cx - s, 0):cx + s] = 1
        elif k == 2: im[(yy >= cy - s) & (yy <= cy + s) & (np.abs(xx - cx) <= (yy - cy + s) / 2)] = 1
        else: im[cy - 1:cy + 2, max(cx - s, 0):cx + s] = 1; im[max(cy - s, 0):cy + s, cx - 1:cx + 2] = 1
        return np.clip(im + 0.15 * r.normal(size=im.shape), 0, 1)
    ks = r.integers(0, 4, 15000)
    allx = torch.tensor(np.stack([shape(k) for k in ks]).astype(np.float32)).unsqueeze(1); ally = torch.tensor(ks)
    Xa, ya, Xb, yb = allx[:12000], ally[:12000], allx[12000:], ally[12000:]
    BENCH = "generated shapes"
n_classes = int(ya.max()) + 1
mlp = nn.Sequential(nn.Flatten(), nn.Linear(28 * 28, 128), nn.ReLU(), nn.Linear(128, n_classes))
cnn = nn.Sequential(nn.Conv2d(1, 16, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2), nn.Conv2d(16, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
                    nn.Flatten(), nn.Linear(32 * 7 * 7, n_classes))
for name, model in [("MLP", mlp), ("CNN", cnn)]:
    fit(model, Xa, ya, epochs=2, lr=1e-3, batch=128)
    print(f"{BENCH}, {name}: test accuracy {accuracy(model, Xb, yb):.3f}, parameters {sum(p.numel() for p in model.parameters()):,}")

## 6. Transfer learning with a pretrained ResNet-18

A ResNet-18 pretrained on ImageNet [3, 5] is loaded from torchvision. Its convolutional layers are frozen, and only a new final layer is trained on 600 crack images, resized to 112 by 112 and repeated over three channels [4]. If the pretrained weights cannot be downloaded, the section is skipped.

In [ ]:
try:
    from torchvision.models import resnet18, ResNet18_Weights
    base = resnet18(weights=ResNet18_Weights.DEFAULT)
    for p in base.parameters():
        p.requires_grad = False
    base.fc = nn.Linear(base.fc.in_features, 2)
    prep = lambda x: nn.functional.interpolate(x, size=112, mode="bilinear", align_corners=False).repeat(1, 3, 1, 1)
    Xs, ys = prep(Xtr[:600]), ytr[:600]
    fit(base, Xs, ys, epochs=3, lr=1e-2, batch=50)
    print("ResNet-18, frozen features, test accuracy:", accuracy(base, prep(Xte), yte))
except Exception as error:
    print("Pretrained weights unavailable, transfer learning skipped:", type(error).__name__)

## Next steps

Take the [self-assessment](https://utkukose.github.io/ai-applications-in-engineering-NB-lecture/weeks/week-11/lab.html#check) in the interactive lab, then answer the [reflection prompts](https://utkukose.github.io/ai-applications-in-engineering-NB-lecture/weeks/week-11/lab.html#reflect) and export the learning log. The [discipline challenges](https://github.com/utkukose/ai-applications-in-engineering-NB-lecture/blob/main/weeks/week-11/README.md#discipline-challenges), the [weekly task](https://github.com/utkukose/ai-applications-in-engineering-NB-lecture/blob/main/weeks/week-11/README.md#weekly-task) and the [research assignment](https://github.com/utkukose/ai-applications-in-engineering-NB-lecture/blob/main/weeks/week-11/README.md#research-and-report-assignment-optional) are listed in the week overview, and [Week 12](https://github.com/utkukose/ai-applications-in-engineering-NB-lecture/blob/main/weeks/week-12/README.md) starts with its overview.

## References

[1] Selvaraju, R. R., Cogswell, M., Das, A., Vedantam, R., Parikh, D., & Batra, D. (2017). Grad-CAM: Visual explanations from deep networks via gradient-based localization. In *2017 IEEE International Conference on Computer Vision (ICCV)* (pp. 618-626). <https://doi.org/10.1109/ICCV.2017.74>

[2] Xiao, H., Rasul, K., & Vollgraf, R. (2017). Fashion-MNIST: A novel image dataset for benchmarking machine learning algorithms. arXiv preprint arXiv:1708.07747. <https://arxiv.org/abs/1708.07747>

[3] He, K., Zhang, X., Ren, S., & Sun, J. (2016). Deep residual learning for image recognition. In *2016 IEEE Conference on Computer Vision and Pattern Recognition (CVPR)* (pp. 770-778). <https://doi.org/10.1109/CVPR.2016.90>

[4] Yosinski, J., Clune, J., Bengio, Y., & Lipson, H. (2014). How transferable are features in deep neural networks?. In *Advances in Neural Information Processing Systems 27* (pp. 3320-3328). <https://arxiv.org/abs/1411.1792>

[5] Deng, J., Dong, W., Socher, R., Li, L.-J., Li, K., & Fei-Fei, L. (2009). ImageNet: A large-scale hierarchical image database. In *2009 IEEE Conference on Computer Vision and Pattern Recognition* (pp. 248-255). <https://doi.org/10.1109/CVPR.2009.5206848>